In [2]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("expense_analysis").getOrCreate()
print("spark session ready")

spark session ready


In [12]:
expenses="""expense_id,user_id,category_id,payment_mode,amount,expense_date
1,1,1,UPI,250,2026-05-01
2,1,2,CASH,12000,2026-05-02
3,1,4,CARD,1500,2026-05-05
4,2,1,CARD,300,2026-05-01
5,2,6,UPI,2000,2026-05-03
6,1,3,UPI,400,2026-06-01
7,1,5,CASH,700,2026-06-06
8,2,3,CARD,999,2026-06-05
9,3,1,UPI,450,2026-05-10
10,3,2,BANK_TRANSFER,13000,2026-05-02
11,3,4,CARD,2200,2026-05-18
12,4,6,UPI,1800,2026-05-11
13,4,1,CASH,350,2026-05-13
14,4,3,CARD,500,2026-06-02
15,2,2,BANK_TRANSFER,12500,2026-06-02
16,2,5,CASH,900,2026-06-08
17,5,1,UPI,600,2026-05-06
18,5,4,CARD,2500,2026-05-15
19,5,6,CASH,3000,2026-05-21
20,5,3,UPI,450,2026-06-03
21,1,6,CARD,850,2026-06-07
22,1,4,CARD,3000,2026-06-15
23,3,5,CASH,750,2026-06-10
24,3,6,UPI,2100,2026-06-12
25,4,2,BANK_TRANSFER,14000,2026-06-01
26,4,4,CARD,5000,2026-06-14
27,2,1,UPI,200,2026-07-01
28,2,3,CARD,450,2026-07-02
29,3,6,UPI,60000,2026-07-05
30,1,2,BANK_TRANSFER,45000,2026-07-06"""

with open("expenses.csv","w") as f:
    f.write(expenses)
expense_df = spark.read.csv("expenses.csv", header=True, inferSchema= True)
expense_df.show()


+----------+-------+-----------+-------------+------+------------+
|expense_id|user_id|category_id| payment_mode|amount|expense_date|
+----------+-------+-----------+-------------+------+------------+
|         1|      1|          1|          UPI|   250|  2026-05-01|
|         2|      1|          2|         CASH| 12000|  2026-05-02|
|         3|      1|          4|         CARD|  1500|  2026-05-05|
|         4|      2|          1|         CARD|   300|  2026-05-01|
|         5|      2|          6|          UPI|  2000|  2026-05-03|
|         6|      1|          3|          UPI|   400|  2026-06-01|
|         7|      1|          5|         CASH|   700|  2026-06-06|
|         8|      2|          3|         CARD|   999|  2026-06-05|
|         9|      3|          1|          UPI|   450|  2026-05-10|
|        10|      3|          2|BANK_TRANSFER| 13000|  2026-05-02|
|        11|      3|          4|         CARD|  2200|  2026-05-18|
|        12|      4|          6|          UPI|  1800|  2026-05

In [14]:
from pyspark.sql.functions import (
    col, sum, avg, max, month, year, to_date, regexp_replace, when
)
expense_df.withColumn("expense_date", to_date(col("expense_date")))
expense_df.show()

+----------+-------+-----------+-------------+------+------------+
|expense_id|user_id|category_id| payment_mode|amount|expense_date|
+----------+-------+-----------+-------------+------+------------+
|         1|      1|          1|          UPI|   250|  2026-05-01|
|         2|      1|          2|         CASH| 12000|  2026-05-02|
|         3|      1|          4|         CARD|  1500|  2026-05-05|
|         4|      2|          1|         CARD|   300|  2026-05-01|
|         5|      2|          6|          UPI|  2000|  2026-05-03|
|         6|      1|          3|          UPI|   400|  2026-06-01|
|         7|      1|          5|         CASH|   700|  2026-06-06|
|         8|      2|          3|         CARD|   999|  2026-06-05|
|         9|      3|          1|          UPI|   450|  2026-05-10|
|        10|      3|          2|BANK_TRANSFER| 13000|  2026-05-02|
|        11|      3|          4|         CARD|  2200|  2026-05-18|
|        12|      4|          6|          UPI|  1800|  2026-05

In [15]:
expense_df.withColumn("Year", year(col("expense_date"))).show()
expense_df= expense_df.withColumn("Month", month(col("expense_date")))
expense_df.show()

+----------+-------+-----------+-------------+------+------------+----+
|expense_id|user_id|category_id| payment_mode|amount|expense_date|Year|
+----------+-------+-----------+-------------+------+------------+----+
|         1|      1|          1|          UPI|   250|  2026-05-01|2026|
|         2|      1|          2|         CASH| 12000|  2026-05-02|2026|
|         3|      1|          4|         CARD|  1500|  2026-05-05|2026|
|         4|      2|          1|         CARD|   300|  2026-05-01|2026|
|         5|      2|          6|          UPI|  2000|  2026-05-03|2026|
|         6|      1|          3|          UPI|   400|  2026-06-01|2026|
|         7|      1|          5|         CASH|   700|  2026-06-06|2026|
|         8|      2|          3|         CARD|   999|  2026-06-05|2026|
|         9|      3|          1|          UPI|   450|  2026-05-10|2026|
|        10|      3|          2|BANK_TRANSFER| 13000|  2026-05-02|2026|
|        11|      3|          4|         CARD|  2200|  2026-05-1

In [16]:
monthly_df= expense_df.groupby("user_id", "Month").agg(sum("amount").alias("Monthly_expense"))
monthly_df.show()

+-------+-----+---------------+
|user_id|Month|Monthly_expense|
+-------+-----+---------------+
|      1|    7|          45000|
|      5|    6|            450|
|      3|    5|          15650|
|      2|    5|           2300|
|      4|    6|          19500|
|      3|    6|           2850|
|      1|    5|          13750|
|      1|    6|           4950|
|      3|    7|          60000|
|      2|    7|            650|
|      4|    5|           2150|
|      5|    5|           6100|
|      2|    6|          14399|
+-------+-----+---------------+



In [18]:
#unusual spikes
user_avg_spend= monthly_df.groupby("user_id").agg(avg("Monthly_expense").alias("Avg_monthly_expense"))
spend_with_avg = monthly_df.join(user_avg_spend, on="user_id")

spike_df= spend_with_avg.withColumn("unusual_spike",
                                    when( col("monthly_expense") > (1.5 * col("Avg_monthly_expense")), "Yes").otherwise("No"))
spike_df.show()

+-------+-----+---------------+-------------------+-------------+
|user_id|Month|Monthly_expense|Avg_monthly_expense|unusual_spike|
+-------+-----+---------------+-------------------+-------------+
|      1|    7|          45000| 21233.333333333332|          Yes|
|      5|    6|            450|             3275.0|           No|
|      3|    5|          15650| 26166.666666666668|           No|
|      2|    5|           2300|             5783.0|           No|
|      4|    6|          19500|            10825.0|          Yes|
|      3|    6|           2850| 26166.666666666668|           No|
|      1|    5|          13750| 21233.333333333332|           No|
|      1|    6|           4950| 21233.333333333332|           No|
|      3|    7|          60000| 26166.666666666668|          Yes|
|      2|    7|            650|             5783.0|           No|
|      4|    5|           2150|            10825.0|           No|
|      5|    5|           6100|             3275.0|          Yes|
|      2| 

In [20]:
#One-time large expense
large_expense= expense_df.filter(col("amount") > 5000)\
               .select("expense_id", "user_id", "category_id", "amount", "expense_date")\
               .withColumn("flag", col("amount"))
print("Large one-time expenses ( >5000 )")
large_expense.show()

Large one-time expenses ( >5000 )
+----------+-------+-----------+------+------------+-----+
|expense_id|user_id|category_id|amount|expense_date| flag|
+----------+-------+-----------+------+------------+-----+
|         2|      1|          2| 12000|  2026-05-02|12000|
|        10|      3|          2| 13000|  2026-05-02|13000|
|        15|      2|          2| 12500|  2026-06-02|12500|
|        25|      4|          2| 14000|  2026-06-01|14000|
|        29|      3|          6| 60000|  2026-07-05|60000|
|        30|      1|          2| 45000|  2026-07-06|45000|
+----------+-------+-----------+------+------------+-----+



In [21]:
#users with unusual spending
spike_df.filter(col("unusual_spike") == "Yes")\
        .select("user_id", "Month","monthly_expense", "Avg_monthly_expense")\
        .show()

+-------+-----+---------------+-------------------+
|user_id|Month|monthly_expense|Avg_monthly_expense|
+-------+-----+---------------+-------------------+
|      1|    7|          45000| 21233.333333333332|
|      4|    6|          19500|            10825.0|
|      3|    7|          60000| 26166.666666666668|
|      5|    5|           6100|             3275.0|
|      2|    6|          14399|             5783.0|
+-------+-----+---------------+-------------------+

